# Demo

Zeigt an einzelnen Anfragebildern, was das Retrieval tatsaechlich liefert:
die aehnlichsten Datenbankbilder, ihre Entfernung zur echten Position, und
die Verteilung der Treffer auf der Karte.

Setzt voraus, dass 06_retrieval fuer das aktuelle Verfahren gelaufen ist.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.rcParams["figure.dpi"] = 150


PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config, paths
from src.geo import haversine_distance
from src.quellen import notiere_quellen
from src.run_guard import (
    embedding_fingerprint,
    print_run_header,
    require_fingerprint,
    validate_config,
)

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)
validate_config(CFG)

METHOD = CFG["vpr"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
EMBEDDING_NAME = embedding_name(CFG)

IMAGE_PATH = PATHS.images
EMBEDDING_DIR = PATHS.embedding_dir(METHOD)
FIGURE_DIR = PATHS.figures / "demo"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"
embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
retrieval_path = PATHS.retrieval_file(EMBEDDING_NAME, METHOD)

embedding_metadata = pd.read_parquet(metadata_path)
FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)
require_fingerprint(embedding_path, FINGERPRINT, what="Embeddings")
require_fingerprint(retrieval_path, FINGERPRINT, what="Retrieval-Ergebnis")

retrieval = np.load(retrieval_path)
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]

database_metadata = embedding_metadata[
    (embedding_metadata["split"] == "database").to_numpy()
].reset_index(drop=True)
query_metadata = embedding_metadata[
    (embedding_metadata["split"] == "query").to_numpy()
].reset_index(drop=True)

# Die Bilder liegen ausserhalb des Repos. Fehlen sie, zeichnet matplotlib
# jede Abbildung dieses Notebooks trotzdem -- leere Kaesten mit farbigem
# Rahmen, Titel und Distanzangabe. Die Datei sieht dann richtig aus und ist
# es nicht; genau so ist eine leere vergleich_query*.png ins Git geraten.
# Lieber hier abbrechen als eine Abbildung speichern, der man das nicht ansieht.
if not any(IMAGE_PATH.glob("*.jpg")):
    raise FileNotFoundError(
        f"Keine Bilder unter {IMAGE_PATH}.\n"
        "Bilder liegen ausserhalb des Repos (03_image_download). Je Rechner:\n"
        "  VPR_IMAGE_ROOT=/pfad/zu/mapillary  oder  VPR_IMAGE_PATH=/pfad/zur/stadt"
    )

print_run_header(CFG, "demo")
print(f"Queries:    {len(query_metadata):,}")
print(f"Database:   {len(database_metadata):,}")
print(f"Bildordner: {IMAGE_PATH}")

In [ ]:
db_lat = database_metadata["lat"].to_numpy()
db_lon = database_metadata["lon"].to_numpy()


def treffer_abstaende(query_index, k):
    """Entfernung der ersten k Treffer zur echten Position der Query."""
    q = query_metadata.iloc[query_index]
    idx = retrieved_indices[query_index][:k]
    d = haversine_distance(q["lat"], q["lon"], db_lat[idx], db_lon[idx])
    return idx, d, similarities[query_index][:k]


def lade_bild(image_id):
    pfad = IMAGE_PATH / f"{image_id}.jpg"
    if not pfad.exists():
        return None
    with Image.open(pfad) as im:
        im.draft("RGB", (512, 512))
        return im.convert("RGB")


## Treffer eines Anfragebildes

Links das Anfragebild, rechts die aehnlichsten Datenbankbilder. Gruen
umrandet heisst: liegt innerhalb der Schwelle, zaehlt also als Treffer.

In [ ]:
def zeige_treffer(query_index, k=5, schwelle=25.0, speichern=None):
    q = query_metadata.iloc[query_index]
    idx, dist, sim = treffer_abstaende(query_index, k)

    fig, achsen = plt.subplots(1, k + 1, figsize=(2.4 * (k + 1), 3.2))

    bild = lade_bild(q["image_id"])
    if bild is not None:
        achsen[0].imshow(bild)
    else:
        achsen[0].text(0.5, 0.5, "Bild fehlt", ha="center")
    achsen[0].set_title("Anfrage", fontsize=9, weight="bold")
    achsen[0].axis("off")

    for platz, (di, d, s) in enumerate(zip(idx, dist, sim), start=1):
        ax = achsen[platz]
        bild = lade_bild(database_metadata.iloc[di]["image_id"])
        if bild is not None:
            ax.imshow(bild)
        else:
            ax.text(0.5, 0.5, "Bild fehlt", ha="center")
        ax.set_title(f"#{platz}   {d:,.0f} m\ncos {s:.3f}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        farbe = "#2e7d32" if d <= schwelle else "#c62828"
        for rand in ax.spines.values():
            rand.set_edgecolor(farbe)
            rand.set_linewidth(2.5)

    fig.suptitle(
        f"{EMBEDDING_NAME}  |  Query {query_index}  |  "
        f"bester Treffer {dist.min():,.0f} m  |  Schwelle {schwelle:g} m",
        fontsize=10,
    )
    plt.tight_layout()
    if speichern:
        fig.savefig(FIGURE_DIR / speichern, bbox_inches="tight")
        # CC BY-SA 4.0: Urheber je Bild -- als Link auf die Bildseite, in QUELLEN.md.
        notiere_quellen(FIGURE_DIR, speichern,
                        [q["image_id"], *database_metadata.iloc[idx]["image_id"]])
        print(f"gespeichert: {FIGURE_DIR / speichern}")
    plt.show()


zeige_treffer(0, k=5)

## Interessante Faelle statt Zufall

Ein zufaellig gewaehltes Anfragebild sagt wenig. Diese Zelle sucht drei
Sorten heraus: einen klaren Erfolg, einen klaren Fehlschlag, und einen Fall,
bei dem die Treffer sich auf zwei weit auseinanderliegende Orte verteilen --
der Grund, warum ein Schwerpunkt ueber alle Treffer in die Irre fuehrt.

In [ ]:
def finde_beispiele(k=10, eng=10.0, weit=200.0):
    erfolg, fehlschlag, gespalten = [], [], []

    for qi in range(len(query_metadata)):
        _, dist, _ = treffer_abstaende(qi, k)
        if dist[0] <= eng:
            erfolg.append((dist[0], qi))
        if dist.min() > weit:
            fehlschlag.append((dist.min(), qi))
        # Erster Treffer nah, aber die Spanne insgesamt gross: zwei Gruppen
        if dist[0] <= eng and dist.max() > weit:
            gespalten.append((dist.max(), qi))

    erfolg.sort()
    fehlschlag.sort()
    gespalten.sort(reverse=True)
    return (
        [qi for _, qi in erfolg[:5]],
        [qi for _, qi in fehlschlag[:5]],
        [qi for _, qi in gespalten[:5]],
    )


erfolge, fehlschlaege, gespaltene = finde_beispiele()
print(f"klare Erfolge:      {erfolge}")
print(f"klare Fehlschlaege: {fehlschlaege}")
print(f"zwei Gruppen:       {gespaltene}")

In [ ]:
if erfolge:
    zeige_treffer(erfolge[0], k=5, speichern=f"{EMBEDDING_NAME}_erfolg.png")
if fehlschlaege:
    zeige_treffer(fehlschlaege[0], k=5, speichern=f"{EMBEDDING_NAME}_fehlschlag.png")
if gespaltene:
    zeige_treffer(gespaltene[0], k=5, speichern=f"{EMBEDDING_NAME}_zwei_gruppen.png")

## Karte

Echte Position der Anfrage gegen die Positionen der Treffer. Beim
gespaltenen Fall sieht man unmittelbar, warum ein Mittelwert ueber alle
Treffer an einem Ort landen wuerde, den kein einziger Treffer stuetzt.

In [ ]:
import folium


def karte(query_index, k=10, schwelle=25.0):
    q = query_metadata.iloc[query_index]
    idx, dist, sim = treffer_abstaende(query_index, k)

    m = folium.Map(location=[q["lat"], q["lon"]], zoom_start=15)
    folium.Marker(
        [q["lat"], q["lon"]],
        tooltip="echte Position der Anfrage",
        icon=folium.Icon(color="blue", icon="camera"),
    ).add_to(m)

    for platz, (di, d, s) in enumerate(zip(idx, dist, sim), start=1):
        zeile = database_metadata.iloc[di]
        folium.CircleMarker(
            [zeile["lat"], zeile["lon"]],
            radius=6,
            color="#2e7d32" if d <= schwelle else "#c62828",
            fill=True,
            tooltip=f"#{platz}: {d:,.0f} m, cos {s:.3f}",
        ).add_to(m)
        folium.PolyLine(
            [[q["lat"], q["lon"]], [zeile["lat"], zeile["lon"]]],
            weight=1,
            opacity=0.4,
        ).add_to(m)

    return m


karte(gespaltene[0] if gespaltene else 0)

## Auf dem Straßennetz

Dieselbe Karte als Bild statt als interaktive Seite — für Folien und die
Ausarbeitung. Das Straßennetz kommt aus OpenStreetMap über osmnx, gecacht
unter `cache/` wie in 01. Blau die echte Position der Anfrage, grün die
Treffer innerhalb der Schwelle, rot die daneben; die Zahl ist der Rang.

In [ ]:
import osmnx as ox

from src.districts import city_boundary, configure_osmnx

# Cache und Overpass-Einstellungen wie ueberall aus config.yaml -> osm.
configure_osmnx(CFG, PATHS.cache)

# Stadtgrenze zur Orientierung -- dieselbe Abfrage wie in 01, aus dem Cache.
# Ueber city_boundary statt ox.geocode_to_gdf(...).iloc[0]: der Geocoder
# liefert je nach Stadt auch Punkte und den gleichnamigen Landkreis, und das
# erste Ergebnis ist nicht zwingend die Stadtflaeche. city_boundary filtert
# erst auf Polygone -- sonst zeigt die Karte hier eine andere Grenze als 01.
STADTGRENZE, _ = city_boundary(CFG["city"])


def karte_strassennetz(query_index, k=10, schwelle=25.0, radius_m=None, speichern=None):
    q = query_metadata.iloc[query_index]
    idx, dist, sim = treffer_abstaende(query_index, k)
    lat = np.r_[q["lat"], db_lat[idx]]
    lon = np.r_[q["lon"], db_lon[idx]]

    # Ausschnitt so gross, dass auch der entfernteste Treffer draufpasst.
    # graph_from_point nimmt dist als halbe Kantenlaenge des Quadrats.
    if radius_m is None:
        radius_m = max(300.0, float(dist.max()) * 1.15)
    G = ox.graph_from_point((q["lat"], q["lon"]), dist=radius_m,
                            network_type="drive", simplify=True)

    fig, ax = ox.plot_graph(G, node_size=0, edge_color="0.75", edge_linewidth=0.8,
                            bgcolor="white", show=False, close=False, figsize=(7, 7))
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    for teil in getattr(STADTGRENZE, "geoms", [STADTGRENZE]):
        ax.plot(*teil.exterior.xy, color="#1565c0", linewidth=1.2, alpha=0.6,
                linestyle="--", zorder=2)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    for platz, (la, lo, d) in enumerate(zip(lat[1:], lon[1:], dist), start=1):
        farbe = "#2e7d32" if d <= schwelle else "#c62828"
        ax.plot([q["lon"], lo], [q["lat"], la], color=farbe, linewidth=0.6, alpha=0.5, zorder=3)
        ax.scatter(lo, la, s=90, color=farbe, edgecolor="white", linewidth=0.8, zorder=4)
        ax.annotate(str(platz), (lo, la), fontsize=7, ha="center", va="center",
                    color="white", weight="bold", zorder=5)
    # Hohler Stern unter den Treffern: die richtigen liegen meist so dicht an
    # der Anfrage, dass ein gefuellter Stern sie verdecken wuerde.
    ax.scatter(q["lon"], q["lat"], marker="*", s=520, facecolor="none",
               edgecolor="#1565c0", linewidth=2.0, zorder=2)
    ax.set_title(f"{EMBEDDING_NAME}  |  Query {query_index}  |  "
                 f"Top-{k}, {int((dist <= schwelle).sum())} innerhalb {schwelle:g} m  |  "
                 f"entferntester Treffer {dist.max():,.0f} m",
                 fontsize=9)
    if speichern:
        fig.savefig(FIGURE_DIR / speichern, bbox_inches="tight", dpi=150)
    plt.show()


if gespaltene:
    karte_strassennetz(gespaltene[0], speichern=f"{EMBEDDING_NAME}_karte_zwei_gruppen.png")
if fehlschlaege:
    karte_strassennetz(fehlschlaege[0], speichern=f"{EMBEDDING_NAME}_karte_fehlschlag.png")

## Encoder im Vergleich

Dasselbe Anfragebild, die Top-5 jedes Encoders untereinander. Geladen wird,
was unter `results/retrieval/` liegt — Baselines, Adapter- und abgeleitete
Varianten. Die Encoder halten dieselben Bilder nicht zwingend in derselben
Reihenfolge (auf zwei Rechnern gerechnet), deshalb wird über die `image_id`
zusammengeführt, nie über die Zeilennummer.

In [ ]:
def lade_alle_retrievals():
    """
    embedding_name -> (indices, similarities, query_row, db_image_ids).

    query_row bildet image_id -> Zeile in der Trefferliste dieses Encoders,
    db_image_ids bildet Zeile -> image_id der Datenbank. Damit meint ein
    Anfragebild ueberall dasselbe, auch wenn die Reihenfolge abweicht.
    """
    ergebnis = {}
    for npz in sorted(PATHS.retrieval.glob("*/*_retrieval.npz")):
        name = npz.name.replace("_retrieval.npz", "")
        meta_pfad = PATHS.metadata_file(name, npz.parent.name)
        if not meta_pfad.exists() or not npz.with_name(npz.name + ".fingerprint.json").exists():
            continue
        meta = pd.read_parquet(meta_pfad, columns=["image_id", "split"])
        q_ids = meta.loc[meta["split"] == "query", "image_id"].to_numpy()
        db_ids = meta.loc[meta["split"] == "database", "image_id"].to_numpy()
        r = np.load(npz)
        ergebnis[name] = (
            r["indices"], r["similarities"],
            pd.Series(np.arange(len(q_ids)), index=q_ids),
            db_ids,
        )
    return ergebnis


ALLE = lade_alle_retrievals()
DB_POS = pd.Series(np.arange(len(database_metadata)), index=database_metadata["image_id"].to_numpy())
print(f"{len(ALLE)} Retrieval-Ergebnisse: {', '.join(ALLE)}")


# Standard: die echten Encoder ohne Adapter. Abgeleitete Varianten und
# Adapter ueber nur=[...] anfordern, sonst werden es 34 Zeilen.
BASIS = [m for m in CFG["vpr"]["models"]
         if m in ALLE and not ({"source", "sources"} & set(CFG["vpr"].get(m) or {}))]


def vergleiche_encoder(query_index, k=5, schwelle=25.0, nur=None, speichern=None):
    namen = [n for n in ALLE if n in (BASIS if nur is None else nur)]
    q = query_metadata.iloc[query_index]
    fig, achsen = plt.subplots(len(namen), k + 1,
                               figsize=(2.2 * (k + 1), 2.3 * len(namen)), squeeze=False)

    # Wieviele Kacheln wirklich ein Bild bekommen haben. Ohne diesen Zaehler
    # zeichnet die Abbildung bei fehlenden Bildern leere Kaesten mit farbigem
    # Rahmen und Titel -- sie sieht richtig aus und ist es nicht. Genau so ist
    # eine leere vergleich_query*.png ins Git geraten.
    geladen = 0
    gezeigt = [q["image_id"]]

    bild_q = lade_bild(q["image_id"])
    geladen += bild_q is not None
    for zeile, name in enumerate(namen):
        indices, sims, query_row, db_ids = ALLE[name]
        if q["image_id"] not in query_row.index:
            continue
        r = query_row[q["image_id"]]
        # Zeilen dieses Encoders -> image_id -> Zeilen unserer database_metadata
        idx = DB_POS[db_ids[indices[r][:k]]].to_numpy()
        sim = sims[r][:k]
        dist = haversine_distance(q["lat"], q["lon"], db_lat[idx], db_lon[idx])
        gezeigt += list(database_metadata.iloc[idx]["image_id"])

        ax = achsen[zeile, 0]
        if bild_q is not None:
            ax.imshow(bild_q)
        ax.set_ylabel(name, fontsize=9, weight="bold", rotation=0, ha="right", va="center")
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)

        for platz, (di, d, s) in enumerate(zip(idx, dist, sim), start=1):
            ax = achsen[zeile, platz]
            bild = lade_bild(database_metadata.iloc[di]["image_id"])
            if bild is not None:
                ax.imshow(bild)
                geladen += 1
            farbe = "#2e7d32" if d <= schwelle else "#c62828"
            for sp in ax.spines.values():
                sp.set_edgecolor(farbe)
                sp.set_linewidth(3)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{d:,.0f} m  cos {s:.2f}", fontsize=7, color=farbe)

    achsen[0, 0].set_title("Anfrage", fontsize=9, weight="bold")
    fig.suptitle(f"Query {query_index}  |  Top-{k} je Encoder, grün = innerhalb {schwelle:g} m", fontsize=10)
    plt.tight_layout()
    if geladen == 0:
        plt.close(fig)
        raise FileNotFoundError(
            f"Kein einziges Bild geladen -- erwartet unter {IMAGE_PATH}.\n"
            "Die Abbildung waere ein Raster leerer Kaesten. Bilder liegen "
            "ausserhalb des Repos; VPR_IMAGE_ROOT oder VPR_IMAGE_PATH setzen."
        )
    if speichern:
        fig.savefig(FIGURE_DIR / speichern, bbox_inches="tight", dpi=150)
        notiere_quellen(FIGURE_DIR, speichern, gezeigt)
    plt.show()


# Ein Fall, den der aktuelle Encoder loest -- sehen die anderen ihn auch?
if erfolge:
    vergleiche_encoder(erfolge[0], speichern=f"vergleich_query{erfolge[0]}.png")

## Eigenes Bild testen

Ein beliebiges Foto durch denselben Encoder schicken und fragen, wo das
System es verortet -- ueber `src/locate.py`, denselben Weg wie 04 bis 06
fuer ein Bild. Geht fuer die Basis-Encoder und die abgeleiteten Varianten
(PCA, Whitening, Verkettung), auch fuer AnyLoc: 04 legt die angepasste PCA
als `anyloc_pca.npz` neben die Embeddings, die Factory laedt sie. Von der
Kommandozeile: `python locate.py foto.jpg --method eigenplaces_megaloc_concat`.

In [ ]:
from src.locate import Locator

# Datenbank-Index sofort, der Encoder erst beim ersten Foto.
locator = Locator(CFG, PROJECT_ROOT, METHOD, ADAPTER)

In [ ]:
def wo_ist_das(bildpfad, k=5):
    """Eigenes Foto einbetten und die aehnlichsten Datenbankbilder zeigen."""
    bildpfad = Path(bildpfad).expanduser()
    if not bildpfad.exists():
        raise FileNotFoundError(bildpfad)

    antwort = locator.locate(bildpfad, k=k)
    treffer = antwort["treffer"]

    fig, achsen = plt.subplots(1, k + 1, figsize=(2.4 * (k + 1), 3.2))
    with Image.open(bildpfad) as im:
        achsen[0].imshow(im.convert("RGB"))
    achsen[0].set_title("eigenes Bild", fontsize=9, weight="bold")
    achsen[0].axis("off")
    for platz, t in enumerate(treffer, start=1):
        bild = lade_bild(t["image_id"])
        if bild is not None:
            achsen[platz].imshow(bild)
        else:
            achsen[platz].text(0.5, 0.5, "Bild fehlt", ha="center")
        achsen[platz].set_title(f"#{platz}  cos {t['aehnlichkeit']:.3f}\n{t['lat']:.5f}, {t['lon']:.5f}",
                                fontsize=7)
        achsen[platz].axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Geschaetzte Position: {antwort['lat']:.6f}, {antwort['lon']:.6f}")
    print(f"Konfidenz (cos):      {antwort['konfidenz']:.4f}   Marge zu Platz 2: {antwort['marge']:.4f}")
    print(f"Streuung der Top-{k}:  {antwort['streuung_m']:,.0f} m")
    if antwort["streuung_m"] > 200:
        print("Die Treffer liegen weit auseinander -- eher ein unsicherer Fall.")

    m = folium.Map(location=[antwort["lat"], antwort["lon"]], zoom_start=16)
    for platz, t in enumerate(treffer, start=1):
        folium.CircleMarker([t["lat"], t["lon"]], radius=6, fill=True,
                            tooltip=f"#{platz}: cos {t['aehnlichkeit']:.3f}").add_to(m)
    return m


# Pfad anpassen und ausfuehren:
# wo_ist_das("~/Downloads/mein_foto.jpg", k=5)